# 00 — Ambiente Colab e pipeline de dados

Objetivo: montar o Drive, registrar o ambiente e validar o carregamento de **treino e validação somente**. Este notebook não treina modelos, não abre o teste interno e não acessa o FUSeg.

Runtime de referência: **Colab 2026.07**, Python 3.12, GPU. Se esse runtime não estiver disponível, use o mais recente e preserve o relatório gerado pela célula de diagnóstico.

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    print("Execução fora do Colab: montagem do Drive ignorada.")

# Ajuste somente se a pasta estiver em outro local do seu Google Drive.
PROJECT_ROOT = Path("/content/drive/MyDrive/TCC/Projeto_TCC_Lucas")
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"Raiz do projeto não encontrada: {PROJECT_ROOT}")
os.chdir(PROJECT_ROOT)
print(f"Projeto: {PROJECT_ROOT}")

## Dependências

O PyTorch do Colab não é reinstalado, porque sua distribuição já é compatível com o CUDA do runtime. As bibliotecas adicionais possuem versões fixadas em `requirements-colab.txt`. Após esta célula, reinicie o runtime apenas se o Colab solicitar.

In [ ]:
import subprocess
import sys


def run_and_show(command):
    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(result.stdout, end="")
    print(f"\nreturncode={result.returncode}")
    result.check_returncode()
    return result


_ = run_and_show(
    [sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"],
)

## Registro do ambiente real

In [ ]:
_ = run_and_show(
    [sys.executable, "scripts/check_environment.py"],
)

## Cache temporário dos dados

O dataset ocupa aproximadamente 303 MB. A cópia para o disco temporário do runtime reduz leituras repetidas no Google Drive. O código e todos os resultados permanentes continuam no Drive.

In [ ]:
import shutil

DATA_ROOT = Path("/content/tcc_data")
shutil.copytree(
    PROJECT_ROOT / "Dataset",
    DATA_ROOT / "Dataset",
    dirs_exist_ok=True,
)
print(f'Dados temporários: {DATA_ROOT / "Dataset"}')

## Testes do carregamento

Os testes abaixo usam imagens reais apenas de treino/validação. Também confirmam que a proteção rejeita uma solicitação de partição reservada antes de abrir seu manifesto.

In [ ]:
_ = run_and_show(
    [sys.executable, "-m", "pytest", "-q", "tests/test_development_data.py"],
)

## Smoke test com PyTorch DataLoader

In [ ]:
for split in ("train", "validation"):
    _ = run_and_show(
        [
            sys.executable,
            "scripts/run_data_pipeline_smoke.py",
            "--project-root",
            str(PROJECT_ROOT),
            "--data-root",
            str(DATA_ROOT),
            "--split",
            split,
            "--samples",
            "4",
            "--with-torch-loader",
        ],
    )

## Critério de conclusão

A etapa está pronta quando: o relatório mostra GPU disponível; os testes terminam sem erro; os batches possuem formato `[B, 3, 384, 512]` para imagem e `[B, 1, 384, 512]` para máscara; e as máscaras contêm apenas 0 e 1. A saída do notebook pode ser consultada no Google Drive.